<a href="https://colab.research.google.com/github/dsb4k8/DS-5530_Assignment_2_3_Combined/blob/main/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <h1 align="center">Assignment2</h1>
<h5 align="center">(work in progress)</h5>


### Data link Q1: https://app.box.com/s/jm6pw202asu4xd3uypwtry2rqk691y1i
1) >The provided data (link above) contains various details and attributes associated with used cars. The target variable, which is the central focus of analysis, is the price of the used cars, and it is measured in lakhs. The data in this dataset is tabular, with rows and columns, where each row represents a specific used car listing, and each column represents a particular attribute or feature of these cars. Features are Make and model of the car, Location or city of sale, Year of manufacture, Mileage, Odometer (kilometers driven), Fuel type (petrol or diesel), Transmission type (manual or automatic), Number of owners, Engine displacement, Engine horsepower, Number of seats, and Price when the car was new.

In [304]:
import numpy as np
import pandas as pd

### Getting Data

In [305]:
# import requests
# import os
#
# url = "https://app.box.com/s/jm6pw202asu4xd3uypwtry2rqk691y1i"
# target_dir = "Assignment_2/data_raw"
# filename = "train.csv"
#
# # Ensure the directory exists
# os.makedirs(target_dir, exist_ok=True)
#
# # Download and save the file
# response = requests.get(url)
# with open(os.path.join(target_dir, filename), 'wb') as f:
#     f.write(response.content)

## EDA & Preprocessing

In [306]:
df_raw = pd.read_csv('Assignment_2/data_raw/train.csv')
df_raw
# print(df_raw.head())
# print(df_raw.dtypes)

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5842,6014,Maruti Swift VDI,Delhi,2014,27365,Diesel,Manual,First,28.4 kmpl,1248 CC,74 bhp,5.0,7.88 Lakh,4.75
5843,6015,Hyundai Xcent 1.1 CRDi S,Jaipur,2015,100000,Diesel,Manual,First,24.4 kmpl,1120 CC,71 bhp,5.0,NaN,4.00
5844,6016,Mahindra Xylo D4 BSIV,Jaipur,2012,55000,Diesel,Manual,Second,14.0 kmpl,2498 CC,112 bhp,8.0,NaN,2.90
5845,6017,Maruti Wagon R VXI,Kolkata,2013,46000,Petrol,Manual,First,18.9 kmpl,998 CC,67.1 bhp,5.0,NaN,2.65


# Problems A & B:
##### a) Look for the missing values in all the columns and either impute them (replace with mean, median, or mode) or drop them. Justify your action for this task. _(4 points)_

In [307]:
# Null EDA

# How many records per column contain nulls?
missing_counts = df_raw.isnull().sum()

# What proportion do the nulls represent per column?
missing_pct = (df_raw.isnull().sum() / len(df_raw)) * 100

# Combining for reporting
missing_report = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_pct
}).sort_values(by='Percentage (%)', ascending=False)

print(missing_report)

                   Missing Count  Percentage (%)
New_Price                   5032       86.061228
Seats                         38        0.649906
Engine                        36        0.615700
Power                         36        0.615700
Mileage                        2        0.034206
Unnamed: 0                     0        0.000000
Name                           0        0.000000
Location                       0        0.000000
Year                           0        0.000000
Kilometers_Driven              0        0.000000
Fuel_Type                      0        0.000000
Transmission                   0        0.000000
Owner_Type                     0        0.000000
Price                          0        0.000000


### Justifications for A:

1) New_Price column value is missing in >86% of records. Imputing this much data would basically create a "fake" dataset. On the other hand, dropping 86% of the rows would leave almost no data to analyze. For these reasons I think this column should be dropped all together as it does not represent values a significant portion of the dataset

2) Seats, Engine, Power, and Mileage on the other hand have < 1% null values. I think due to the small size of the missing data, both imputing and dropping are both valid options, but I will opt to impute values to preserve as much data as possible.
- For Engine, Power, and Milage, I will impute median column value for null entries because median handles outliers better than mean
- For Seats, ill impute the mode of the column instead because, as you cant have for example 3.5 seats


3) Lastly, there is an unnamed column which should be dropped - its basically a redundant index that starts at 1 instead of 0. will drop along with New_Price


In [308]:
# 1) Dropping New_Price
df_reshaped = df_raw.copy()
# df_reshaped.columns
df_reshaped.drop(columns=['New_Price', 'Unnamed: 0'], inplace=True)
# print(df_reshaped.dtypes)
# df_reshaped

- Mileage, Engine (CC displacement amount), Power, etc are currently strings which cannot be worked with numerically. Converting these to float values

# BUT FIRST -  Problem B
##### b) Remove the units from some of the attributes and only keep the numerical values (for example remove kmpl from “Mileage”, CC from “Engine”, bhp from “Power”, and lakh from “New_price”). _(4 points)_
#### Justification for doing this step first:
Removing units first before imputing will allow for easier computation / less parsing at runtime

In [309]:
#B)

#converting string numerics into floats
df_reshaped['Mileage'] = df_reshaped['Mileage'].str.extract(r'(\d+\.\d+|\d+)').astype(float)
df_reshaped['Engine'] = df_reshaped['Engine'].str.extract(r'(\d+)').astype(float)
df_reshaped['Power'] = df_reshaped['Power'].str.extract(r'(\d+\.\d+|\d+)').astype(float)
# df_reshaped.dtypes


In [310]:
#2b) Imputing values for type-converted columns

#medians
df_reshaped['Mileage'] = df_reshaped['Mileage'].fillna(df_reshaped['Mileage'].median())
df_reshaped['Engine'] = df_reshaped['Engine'].fillna(df_reshaped['Engine'].median())
df_reshaped['Power'] = df_reshaped['Power'].fillna(df_reshaped['Power'].median())

#mode
if not df_reshaped['Seats'].mode().empty:
    df_reshaped['Seats'] = df_reshaped['Seats'].fillna(df_reshaped['Seats'].mode()[0])

# df_reshaped

# verifying nulls have been handled as expected
print("Missing values after processing:")
print(df_reshaped.isnull().sum())

Missing values after processing:
Name                 0
Location             0
Year                 0
Kilometers_Driven    0
Fuel_Type            0
Transmission         0
Owner_Type           0
Mileage              0
Engine               0
Power                0
Seats                0
Price                0
dtype: int64
